# YuE2 Studio 5.0.0 — Linux GPU Builder

**This builds the actual upstream YuE2 Studio backend and YuE2 engine, not ComfyUI.**

- Upstream: https://github.com/timoncool/YuE2-Studio
- Pinned release: **v5.0.0 (21 November 2026)**
- Kaggle settings: **Accelerator None**, Internet ON.
- Builds the upstream Rust `music-server` and the pinned `yue2.cpp` runtime with CUDA 12.9 (T4 / sm_75) and Vulkan.
- Adds a checksum-verified Cloudflare `cloudflared` binary.\n- Automatically publishes all runtime artifacts as a private Kaggle Dataset using Kaggle API credentials (JJP automatic dataset-builder pattern).
- Does **not** bundle model weights. The Studio downloads and manages models through its own Models / Engines screens.

## Run

1. Add Kaggle secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` if they are not already available to the notebook.
2. Run this notebook from top to bottom.
3. The final cell automatically creates or versions the private Kaggle Dataset `yue2-studio-linux-runtime`. No manual saving, output attachment or dataset creation is required.

The build is substantial and can take a while. It fails loudly if CUDA 12.9 or the pinned Cloudflare binary cannot be verified; it will not silently produce a CPU-only substitute.

The Studio application is MIT-licensed. Individual model weights and optional engines have their own licences; check those before commercial use.

In [ ]:
import os, sys, json, shutil, hashlib, subprocess, textwrap, urllib.request, base64, io, zipfile
from pathlib import Path

ROOT = Path("/kaggle/working/YuE2-Studio")
BUNDLE = Path("/kaggle/working/yue2-studio-bundle")
TMP = Path("/kaggle/temp/yue2-studio-build")
BUNDLE.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)

def run(cmd, **kwargs):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, check=True, **kwargs)

# Kaggle builder: use Ubuntu's packaged Vulkan headers/loader and glslc.
# Downloading the full LunarG SDK is deliberately avoided: its download host can
# return HTTP 403 to Kaggle runners even when Kaggle Internet is enabled.
run(["bash", "-lc", "apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq build-essential pkg-config cmake ninja-build nasm autoconf automake libtool curl ca-certificates git unzip xz-utils libssl-dev libasound2-dev libvulkan-dev glslc patchelf file ccache clang libclang-dev libgtk-3-dev libayatana-appindicator3-dev"])
glslc = shutil.which("glslc")
if not glslc:
    raise RuntimeError("glslc was not installed from the Ubuntu package repository; check Kaggle Internet and apt sources.")
print("Using system Vulkan development files and shader compiler:", glslc, flush=True)

# CUDA 12.9 includes T4 (sm_75). Hide any preinstalled nvcc so upstream's build script
# does not mistake Kaggle's CUDA 12.x compiler for CUDA 13.
run(["bash", "-lc", "curl -fsSL -o /tmp/cuda-keyring.deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb && dpkg -i /tmp/cuda-keyring.deb >/dev/null && apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq cuda-nvcc-12-9 cuda-cudart-dev-12-9 libcublas-dev-12-9"])
cuda_root = Path("/usr/local/cuda-12.9")
if not (cuda_root / "bin/nvcc").exists():
    raise RuntimeError("CUDA 12.9 nvcc was not installed; stopping rather than producing a CPU-only T4 build.")

# Rust stable.
if not Path("/root/.cargo/bin/cargo").exists():
    run(["bash", "-lc", "curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain stable"])
env = os.environ.copy()
env["PATH"] = f"/root/.cargo/bin:{vulkan_root}/x86_64/bin:/usr/local/bin:/usr/bin:/bin:/usr/sbin:/sbin"
env.pop("VULKAN_SDK", None)  # Let CMake FindVulkan use the apt-installed system SDK.
env["CUDA12_ROOT"] = str(cuda_root)
env["YUE_CUDA_PROFILE"] = "universal"
env["YUE_ENGINE_BUILD_ROOT"] = str(TMP / "engine-build")
env["CARGO_TARGET_DIR"] = str(TMP / "cargo-target")
env["CARGO_BUILD_JOBS"] = "4"
env["YUE_BUILD_JOBS"] = "4"

if ROOT.exists():
    shutil.rmtree(ROOT)
run(["git", "clone", "--depth", "1", "--branch", "v5.0.0", "https://github.com/timoncool/YuE2-Studio.git", str(ROOT)], env=env)

# Build the actual Studio's headless Rust service and its pinned CUDA/Vulkan YuE2 engine.
run(["bash", "-lc", "cargo build --release -p music-server"], cwd=ROOT, env=env)
run(["bash", "-lc", "scripts/build-yue-runtime-linux.sh /kaggle/working/yue2-studio-bundle/yue2-cpp"], cwd=ROOT, env=env)

server = TMP / "cargo-target/release/music-server"
engine = BUNDLE / "yue2-cpp/yue-server"
if not server.is_file() or not engine.is_file():
    raise RuntimeError("Build output is incomplete: music-server or yue-server is missing.")
shutil.copy2(server, BUNDLE / "music-server")
shutil.copy2(ROOT / "LICENSE", BUNDLE / "LICENSE")
shutil.copy2(ROOT / "engines/yue2-cpp-source.json", BUNDLE / "yue2-cpp-source.json")

# Cloudflare Quick Tunnel binary, pinned and checksum-verified.
cf_version = "2026.10.3"
cf_sha256 = "053b80b0a23031a038fdbf3c073f59f82b8f1f152aa67971d7383ffafc0dbab0"
cf_path = BUNDLE / "cloudflared"
cf_url = f"https://github.com/cloudflare/cloudflared/releases/download/{cf_version}/cloudflared-linux-amd64"
urllib.request.urlretrieve(cf_url, cf_path)
digest = hashlib.sha256(cf_path.read_bytes()).hexdigest()
if digest != cf_sha256:
    cf_path.unlink(missing_ok=True)
    raise RuntimeError(f"cloudflared checksum mismatch: {digest}")
cf_path.chmod(0o755)

manifest = {
    "studio_version": "5.0.0",
    "upstream": "https://github.com/timoncool/YuE2-Studio",
    "tag": "v5.0.0",
    "built_for": "Linux x86_64, CUDA 12.9 + Vulkan, NVIDIA T4 and newer supported CUDA devices",
    "engine": json.loads((BUNDLE / "yue2-cpp-source.json").read_text()),
    "cloudflared_version": cf_version,
    "cloudflared_sha256": cf_sha256,
    "components": ["music-server", "yue2-cpp/yue-server", "yue2-cpp CUDA/Vulkan backend libraries", "cloudflared"]
}
(BUNDLE / "manifest.json").write_text(json.dumps(manifest, indent=2))

# --- Automatic Kaggle Dataset Builder (JJP pattern) -------------------------
DATASET_OWNER = ""  # Leave blank to resolve from Kaggle API credentials.
WORK = Path("/kaggle/working/yue2-studio-dataset-builder")
RUNTIME = WORK / "runtime"
RUNTIME.mkdir(parents=True, exist_ok=True)

# Extract the tiny, self-contained dataset helper package from its embedded ZIP.
with zipfile.ZipFile(io.BytesIO(base64.b64decode("UEsDBBQAAAAIAIiVe12CrXMn7QQAAGkMAAARAAAAc3R1ZGlvX2RhdGFzZXQucHmVVttu20YQfddXDLYvVKDQlzYo4EAFFFc2giRuYDsoAsMgKHIkbU1x2d2lbFfQv3dmL6IkX5rqReTs7Nm5nDlLuWiUtqDMAP4yqh6AmbdWVvTfThqtCjS0YnHRTGWFvalWC2hyO6/kBKTf+pVee70Sp5C3do61lUVusczyRib9kx7QTwgx2lqDe2nn8CmfzSqEWlmcKHUHBguN1oDSgA/SWFnPog8tlLw5r0xKWA5TxsDdm9WP/ij+uSjv3NYsogb3bwb1lTedVpIwN5ui4/CpT9LvoDm6epnV+QJB1pCIT6Pz88/j7NvV+PJi9GUsBhBNn8bfRb+LygU95Xwp6JRApFZ1OkObRMA95yd5bf86iJu4/ZZiD1kwbEi9Q3+Cgw8FNhbG7k+q+vmTmtz4Gr/svnF5Pr8nNepzk19x3KuczqVBuGyJAQsca610shOpCDQZff24TRXINcJCGkNMSmFUlrAXB+T1xkZHgthFzc1LHKWJIDKDRt3WabfLV3iLfSnNQBqISI8ZPlisSywjGz38qJFuI3lQBze2wDqG2B6sYKYwWl3zaq93/vEDbTw6PP4F3ryBn/0wFnOlDPG/0Lkt5onGv1tJtckmjxYpgYJyl2Xunqt8glWod2cnyBue7aTpO9o3zPdu+bYX54H1YHdtmxl2ni7uSqmThtpRWzO81i0O/Ihn6s69dtRsTT5D5rFTobSU5i5ztoSROj8imjOnU40Ivw1hN79dJodauTgPfLa9F3g1FRcKQs1gqaqWxnxORFjt4h9QyU/So+kazuUH4BhS0Q9110glyKgOuSFKU8vySs0GoFrbtDajSgyoWPyolbLciGKO3qzua6Q/6mmnm6cOjvkWyRiQQdUwldpYIBK+h6adVNLMIYca72GJ2tCEsk9F2zX7dNLZhUKVdi3uLP09jx9t3gJtnrkKD7fxD0CEeN+yBz+nfM/4QEzVzsg/1OgmumZsF55gVtoKn/NxC8GpJcV2ijz0NWR5+SEN4uFiqSxUPZWzbJlXLSYiwon+tqhF66u6JE5VW5XOX6NR1XLTtrj9PXDzfh9dj67G19kff16ML1PR1ZBSWAmf24lPni4UWdLLVKwixvpgxSVa82VTyQJrg4Y8blbChX0C4vT08O1ReijWt4NdUSuR2C2dfpNfqKqv0PYSAX9vx8dwZdtSKvgs6/aBScSZwqSty4oZv95tfXqvJVHfkswl3OS0bBeNSXidOc+6PDzu+1x/glPVPIKxNMTlBjnXVk7zgu/r2irH+kj3qapK1OlGdeikhVedQAy6g0iaxW3XH6NaXWDkOG+4Ed4mbnekhNvlF1JpMv7SSfbuYt/qM1q5UPZMUQGiXlwGYehy4DjivXMCKw+8Ft2JVGi7PyY+Oq5i4HR0TP3o/R8RDeJZUIGPE3/8wGHtJM0n+sbjAxatzSfEub20XQDFfKHK5FD9+u6dB4gjKEvK4jlaPv0k40GL25YS75MOowvKpcP3DqfzH58cG98zuukxzqm3Pn9sEOYgjYmxelv2BpQIfaOqBX+7wjb3Y181ks6bOY0GqQWhkASprDDLoYuAC1wh4RNN4xnGL3UJErNZlYcggkfWNnxhlr5iSM6vxk7S/jRup/xFjOKF2KTOqIk4FP/IRjwXkD8hBBIuzJXoukRi0b2QOvidZPUPg43Oc1Rk3gtz3fsXUEsBAhQDFAAAAAgAiJV7XYKtcyftBAAAaQwAABEAAAAAAAAAAAAAAIABAAAAAHN0dWRpb19kYXRhc2V0LnB5UEsFBgAAAAABAAEAPwAAABwFAAAAAA=="))) as archive:
    for name in archive.namelist():
        assert Path(name).name == name, f"Unsafe helper path: {name}"
        (RUNTIME / name).write_bytes(archive.read(name))
sys.path.insert(0, str(RUNTIME))
import importlib
importlib.invalidate_caches()
sys.modules.pop("studio_dataset", None)

from studio_dataset import authenticated_api, create_dataset, choose_scratch, GIB

# Catalogue paths are relative to the generated Kaggle Dataset root.
catalog = {
    "dataset_slug": "yue2-studio-linux-runtime",
    "dataset_title": "YuE2 Studio 5.0.0 — Linux GPU Runtime",
    "description": "Pinned upstream YuE2 Studio 5.0.0 Linux music-server, CUDA/Vulkan YuE2 engine and Cloudflare tunnel binary for Kaggle GPU runtime.",
    "assets": [
        {"source": str(BUNDLE / "manifest.json"), "path": "manifest.json"},
        {"source": str(BUNDLE / "music-server"), "path": "music-server", "executable": True},
        {"source": str(BUNDLE / "cloudflared"), "path": "cloudflared", "executable": True},
        {"source": str(BUNDLE / "yue2-cpp-source.json"), "path": "yue2-cpp-source.json"},
        {"source": str(BUNDLE / "yue2-cpp/yue-server"), "path": "yue2-cpp/yue-server", "executable": True},
    ]
}
# Include the complete native engine directory (shared libraries, kernels and support files).
for source in (BUNDLE / "yue2-cpp").rglob("*"):
    if source.is_file() and source.name != "yue-server":
        rel = source.relative_to(BUNDLE)
        catalog["assets"].append({"source": str(source), "path": str(rel), "executable": bool(source.stat().st_mode & 0o111)})

api = authenticated_api()  # Resolve Kaggle secrets before dataset upload begins.
peak = sum(Path(a["source"]).stat().st_size for a in catalog["assets"])
scratch = choose_scratch(peak + 2 * GIB, [Path("/kaggle/temp"), Path("/tmp"), Path("/kaggle/working")], "yue2-studio-dataset-builder")
result = create_dataset(catalog, WORK / "saved-dataset", Path("/kaggle/input"), scratch / "download-cache", DATASET_OWNER, api)

print()
print("AUTOMATIC DATASET PUBLISH COMPLETE")
print(json.dumps(result, indent=2))
print("The Studio Runtime notebook downloads this dataset automatically using the same Kaggle credentials.")
